# Home Credit Default Risk
## From a Strong Feature-Engineering Baseline to Winner-Inspired Experiments

**Competition:** Kaggle — Home Credit Default Risk  
**Primary metric:** ROC-AUC  
**Main model:** LightGBM  
**Best Kaggle private score before winner-study phase:** **0.78868**

### Project objective
This notebook documents an end-to-end tabular machine-learning workflow built incrementally from raw relational data. The goal is not only to obtain a strong score, but to make every modeling decision traceable:

1. build a reproducible baseline;
2. engineer information from each relational table;
3. validate feature families with cross-validation;
4. study top-solution ideas as hypotheses rather than copy them;
5. keep or reject ideas based on controlled experiments.

> **Reproducibility note:** Kaggle competition data is intentionally not embedded in this repository. Put the CSV files in `./data/`, the notebook directory, or set `HOME_CREDIT_DATA_DIR`.

## 1. Experiment history

The solution was developed incrementally. Each major feature family was evaluated before moving to the next one.

| Version | Main change | CV ROC-AUC | Outcome |
|---|---|---:|---|
| V1 | Raw application | 0.759101 | Baseline |
| V2 | EXT engineering | 0.758507 | Rejected |
| V3 | Financial ratios | 0.767108 | Kept |
| V4 | Employment features | 0.766892 | Rejected |
| V5 | Bureau history | 0.771489 | Kept |
| V6 | Bureau balance | 0.772063 | Kept |
| V7 | Previous applications | 0.777159 | Kept |
| V8 | Installment behavior | 0.783466 | Kept |
| V9 | POS_CASH history | 0.785156 | Kept |
| V10 | Credit-card history | 0.787422 | Kept |
| V11 | Recent behavior | 0.789166 | Kept |
| V12 | Recent bureau | 0.791330 | Kept |
| V13 | Robust ratios | 0.791549 | Kept |
| V14A | Cross-table interactions | 0.791589 | Marginal |

### Main finding
The largest gains came from **relational feature engineering**, especially behavioral history and recency, rather than aggressive LightGBM tuning.

## 2. Setup and data loading

The loader below searches common local locations so the notebook can be cloned from GitHub without hard-coding a machine-specific Windows path.

In [ ]:
from pathlib import Path
import os
import gc
import time
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

RANDOM_STATE = 42

def find_data_dir():
    candidates = [
        os.getenv("HOME_CREDIT_DATA_DIR"),
        "./data",
        ".",
        "/kaggle/input/home-credit-default-risk",
    ]
    required = {"application_train.csv", "application_test.csv"}
    for candidate in candidates:
        if not candidate:
            continue
        p = Path(candidate).expanduser().resolve()
        if p.exists() and required.issubset({x.name for x in p.glob("*.csv")}):
            return p
    raise FileNotFoundError(
        "Home Credit CSV files were not found. Put them in ./data, "
        "the notebook folder, or set HOME_CREDIT_DATA_DIR."
    )

DATA_DIR = find_data_dir()
print("Data directory:", DATA_DIR)

def read_csv(name, **kwargs):
    return pd.read_csv(DATA_DIR / name, **kwargs)

application_train = read_csv("application_train.csv")
application_test = read_csv("application_test.csv")
bureau = read_csv("bureau.csv")
bureau_balance = read_csv("bureau_balance.csv")
previous_application = read_csv("previous_application.csv")
installments = read_csv("installments_payments.csv")
pos_cash = read_csv("POS_CASH_balance.csv")
credit_card = read_csv("credit_card_balance.csv")

print("application_train:", application_train.shape)
print("application_test :", application_test.shape)
print("bureau           :", bureau.shape)
print("bureau_balance   :", bureau_balance.shape)
print("previous_app     :", previous_application.shape)
print("installments     :", installments.shape)
print("POS_CASH         :", pos_cash.shape)
print("credit_card      :", credit_card.shape)

## 3. Data understanding and modeling principles

Home Credit is a **relational tabular problem**: one application row can be connected to many bureau credits, previous applications, installments, POS records, and credit-card monthly records.

The central feature-engineering strategy is therefore:

```text
transaction / history rows
        ↓
row-level behavioral signals
        ↓
customer-level aggregation
        ↓
application table
        ↓
LightGBM
```

Important implementation rules used throughout the notebook:

- preserve `SK_ID_CURR` until relational tables have been merged;
- replace invalid infinite ratios with missing values;
- let LightGBM handle missingness instead of indiscriminately filling every `NaN`;
- keep train/test transformations identical;
- use fixed validation splits when comparing experiments.

## 4. Application features

In [ ]:
# ============================================================
# BASELINE — APPLICATION DATA PREPARATION
# ============================================================

# Target
y = application_train["TARGET"].copy()

# Keep IDs separately because we will need them
# to merge relational-table features later
train_ids = application_train["SK_ID_CURR"].copy()
test_ids = application_test["SK_ID_CURR"].copy()


# ============================================================
# START FROM RAW APPLICATION DATA
# ============================================================

X_train_base = application_train.drop(
    columns=["TARGET"]
).copy()

X_test_base = application_test.copy()


print("=" * 70)
print("RAW APPLICATION DATA")
print("=" * 70)

print("Train shape:", X_train_base.shape)
print("Test shape :", X_test_base.shape)

print("\nTarget distribution:")
print(y.value_counts(normalize=True))

In [ ]:
# ============================================================
# BASELINE — APPLICATION FINANCIAL RATIOS
# ============================================================

def add_application_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # Current loan burden relative to income
    # --------------------------------------------------------

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_INCOME_TOTAL"]
    )

    # --------------------------------------------------------
    # Current annual payment burden relative to income
    # --------------------------------------------------------

    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] /
        df["AMT_INCOME_TOTAL"]
    )

    # --------------------------------------------------------
    # Approximate credit duration / structure
    # --------------------------------------------------------

    df["CREDIT_ANNUITY_RATIO"] = (
        df["AMT_CREDIT"] /
        df["AMT_ANNUITY"]
    )

    # --------------------------------------------------------
    # Protect against division by zero
    # --------------------------------------------------------

    ratio_cols = [
        "CREDIT_INCOME_RATIO",
        "ANNUITY_INCOME_RATIO",
        "CREDIT_ANNUITY_RATIO"
    ]

    df[ratio_cols] = (
        df[ratio_cols]
        .replace([np.inf, -np.inf], np.nan)
    )

    return df

In [ ]:
X_train_base = add_application_features(
    X_train_base
)

X_test_base = add_application_features(
    X_test_base
)


print("=" * 70)
print("APPLICATION FEATURES")
print("=" * 70)

print("Train shape:", X_train_base.shape)
print("Test shape :", X_test_base.shape)

display(
    X_train_base[
        [
            "CREDIT_INCOME_RATIO",
            "ANNUITY_INCOME_RATIO",
            "CREDIT_ANNUITY_RATIO"
        ]
    ].describe()
)

In [ ]:
# ============================================================
# CATEGORICAL PREPARATION
# ============================================================

def prepare_categories(train_df, test_df):

    train_df = train_df.copy()
    test_df = test_df.copy()

    categorical_cols = train_df.select_dtypes(
        include=["object"]
    ).columns.tolist()

    for col in categorical_cols:

        # Build a common set of categories from train + test.
        # This guarantees identical categorical codes/categories.
        combined_categories = pd.Index(
            pd.concat(
                [
                    train_df[col],
                    test_df[col]
                ],
                axis=0
            )
            .dropna()
            .unique()
        )

        train_df[col] = pd.Categorical(
            train_df[col],
            categories=combined_categories
        )

        test_df[col] = pd.Categorical(
            test_df[col],
            categories=combined_categories
        )

    return train_df, test_df, categorical_cols

In [ ]:
# ============================================================
# UTILITY — SAFE DIVISION
# ============================================================

def safe_divide(numerator, denominator):

    result = np.where(
        denominator != 0,
        numerator / denominator,
        np.nan
    )

    result = pd.Series(
        result,
        index=numerator.index
    )

    result = result.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return result

## 5. Bureau history

Bureau features summarize external credit exposure, debt, overdue behavior, credit age, active/closed status, and prolongations.

In [ ]:
# ============================================================
# BASELINE — BUREAU FEATURES
# Reproduction of original V5
# ============================================================

def build_bureau_features(bureau_df):

    bureau_df = bureau_df.copy()

    # --------------------------------------------------------
    # Helper for sums
    # --------------------------------------------------------
    # Normal pandas sum can return 0 when every value is NaN.
    #
    # But:
    #   0 debt      != unknown debt
    #
    # min_count=1 keeps a completely missing group as NaN.
    # --------------------------------------------------------

    def sum_with_nan(x):
        return x.sum(min_count=1)


    # ========================================================
    # 1. ROW-LEVEL HELPER FEATURES
    # ========================================================

    bureau_df["IS_ACTIVE"] = (
        bureau_df["CREDIT_ACTIVE"] == "Active"
    ).astype("int8")


    bureau_df["IS_CLOSED"] = (
        bureau_df["CREDIT_ACTIVE"] == "Closed"
    ).astype("int8")


    bureau_df["HAS_OVERDUE"] = (
        bureau_df["CREDIT_DAY_OVERDUE"] > 0
    ).astype("int8")


    # DAYS_CREDIT is negative:
    #
    # -100  = credit opened 100 days ago
    # -1000 = credit opened 1000 days ago
    #
    # Convert it into an intuitive positive age.
    bureau_df["CREDIT_AGE_DAYS"] = (
        -bureau_df["DAYS_CREDIT"]
    )


    # ========================================================
    # 2. AGGREGATE TO ONE ROW PER CUSTOMER
    # ========================================================

    bureau_agg = (
        bureau_df
        .groupby("SK_ID_CURR")
        .agg(

            # -----------------------------------------------
            # Number of previous bureau credits
            # -----------------------------------------------

            BUREAU_CREDIT_COUNT=(
                "SK_ID_BUREAU",
                "count"
            ),


            # -----------------------------------------------
            # Active / closed credits
            # -----------------------------------------------

            BUREAU_ACTIVE_COUNT=(
                "IS_ACTIVE",
                "sum"
            ),

            BUREAU_CLOSED_COUNT=(
                "IS_CLOSED",
                "sum"
            ),


            # -----------------------------------------------
            # Historical credit amounts
            # -----------------------------------------------

            BUREAU_TOTAL_CREDIT=(
                "AMT_CREDIT_SUM",
                sum_with_nan
            ),

            BUREAU_MEAN_CREDIT=(
                "AMT_CREDIT_SUM",
                "mean"
            ),

            BUREAU_MAX_CREDIT=(
                "AMT_CREDIT_SUM",
                "max"
            ),


            # -----------------------------------------------
            # Historical debt
            # -----------------------------------------------

            BUREAU_TOTAL_DEBT=(
                "AMT_CREDIT_SUM_DEBT",
                sum_with_nan
            ),


            # -----------------------------------------------
            # Overdue amount
            # -----------------------------------------------

            BUREAU_TOTAL_OVERDUE=(
                "AMT_CREDIT_SUM_OVERDUE",
                sum_with_nan
            ),


            # -----------------------------------------------
            # Maximum current overdue duration
            # -----------------------------------------------

            BUREAU_MAX_OVERDUE_DAYS=(
                "CREDIT_DAY_OVERDUE",
                "max"
            ),


            # -----------------------------------------------
            # Number of overdue bureau credits
            # -----------------------------------------------

            BUREAU_OVERDUE_COUNT=(
                "HAS_OVERDUE",
                "sum"
            ),


            # -----------------------------------------------
            # Credit history timing
            # -----------------------------------------------

            BUREAU_RECENT_CREDIT_DAYS=(
                "CREDIT_AGE_DAYS",
                "min"
            ),

            BUREAU_MEAN_CREDIT_AGE=(
                "CREDIT_AGE_DAYS",
                "mean"
            ),

            BUREAU_OLDEST_CREDIT_DAYS=(
                "CREDIT_AGE_DAYS",
                "max"
            ),


            # -----------------------------------------------
            # Number of credit prolongations
            # -----------------------------------------------

            BUREAU_PROLONG_COUNT=(
                "CNT_CREDIT_PROLONG",
                sum_with_nan
            )
        )
        .reset_index()
    )


    # ========================================================
    # 3. CUSTOMER-LEVEL RATIOS
    # ========================================================

    bureau_agg["BUREAU_ACTIVE_RATIO"] = (
        bureau_agg["BUREAU_ACTIVE_COUNT"] /
        bureau_agg["BUREAU_CREDIT_COUNT"]
    )


    bureau_agg["BUREAU_DEBT_CREDIT_RATIO"] = (
        bureau_agg["BUREAU_TOTAL_DEBT"] /
        bureau_agg["BUREAU_TOTAL_CREDIT"]
    )


    bureau_agg["BUREAU_ANY_OVERDUE"] = (
        bureau_agg["BUREAU_OVERDUE_COUNT"] > 0
    ).astype("int8")


    # ========================================================
    # 4. CLEAN INFINITE VALUES
    # ========================================================

    bureau_agg.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    return bureau_agg

In [ ]:
# ============================================================
# CREATE BUREAU BASELINE FEATURES
# ============================================================

bureau_features = build_bureau_features(
    bureau
)


print("=" * 70)
print("BUREAU BASELINE FEATURES")
print("=" * 70)

print("Original bureau rows :",
      len(bureau))

print("Unique customers     :",
      bureau["SK_ID_CURR"].nunique())

print("Aggregated customers :",
      len(bureau_features))

print(
    "Number of features   :",
    bureau_features.shape[1] - 1
)

print(
    "Shape                :",
    bureau_features.shape
)

display(
    bureau_features.head()
)

In [ ]:
# ============================================================
# MERGE BUREAU FEATURES INTO TRAIN AND TEST
# ============================================================

print("=" * 70)
print("MERGING BUREAU FEATURES")
print("=" * 70)

print("Before merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


X_test_base = X_test_base.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


print("\nRows preserved:")
print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test :",
    len(X_test_base) == len(application_test)
)

## 6. Bureau-balance history

Bureau-balance features convert monthly bureau statuses into customer-level delinquency history.

In [ ]:
# ============================================================
# BASELINE — BUREAU BALANCE FEATURES
# Reproduction of original V6
# ============================================================

def build_bureau_balance_features(
    bureau_balance_df,
    bureau_df
):

    bb = bureau_balance_df.copy()


    # ========================================================
    # 1. MONTH-LEVEL DELINQUENCY FLAGS
    # ========================================================

    # Any delinquency
    bb["BB_DPD_ANY"] = (
        bb["STATUS"].isin(
            ["1", "2", "3", "4", "5"]
        )
    ).astype("int8")


    # 30+ days delinquent
    bb["BB_DPD_30PLUS"] = (
        bb["STATUS"].isin(
            ["2", "3", "4", "5"]
        )
    ).astype("int8")


    # 90+ days delinquent
    bb["BB_DPD_90PLUS"] = (
        bb["STATUS"].isin(
            ["4", "5"]
        )
    ).astype("int8")


    # Very severe delinquency
    bb["BB_DPD_120PLUS"] = (
        bb["STATUS"] == "5"
    ).astype("int8")


    # --------------------------------------------------------
    # Convert STATUS into ordinal delinquency severity
    #
    # 0 -> 0
    # 1 -> 1
    # ...
    # 5 -> 5
    #
    # C and X are not delinquency levels,
    # therefore they become NaN.
    # --------------------------------------------------------

    status_map = {
        "0": 0,
        "1": 1,
        "2": 2,
        "3": 3,
        "4": 4,
        "5": 5
    }

    bb["BB_SEVERITY"] = (
        bb["STATUS"]
        .map(status_map)
    )


    # ========================================================
    # 2. FIRST LEVEL:
    # MONTHS -> ONE ROW PER BUREAU CREDIT
    # ========================================================

    bb_credit = (
        bb
        .groupby("SK_ID_BUREAU")
        .agg(

            # Number of recorded months
            BB_MONTH_COUNT=(
                "MONTHS_BALANCE",
                "count"
            ),

            # How far back the history extends
            BB_HISTORY_LENGTH=(
                "MONTHS_BALANCE",
                lambda x: -x.min()
            ),

            # Number of delinquent months
            BB_DPD_ANY_COUNT=(
                "BB_DPD_ANY",
                "sum"
            ),

            BB_DPD_30PLUS_COUNT=(
                "BB_DPD_30PLUS",
                "sum"
            ),

            BB_DPD_90PLUS_COUNT=(
                "BB_DPD_90PLUS",
                "sum"
            ),

            BB_DPD_120PLUS_COUNT=(
                "BB_DPD_120PLUS",
                "sum"
            ),

            # Worst status observed for this credit
            BB_MAX_SEVERITY=(
                "BB_SEVERITY",
                "max"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 3. PER-CREDIT DELINQUENCY RATIOS
    # ========================================================

    bb_credit["BB_DPD_ANY_RATIO"] = (
        bb_credit["BB_DPD_ANY_COUNT"] /
        bb_credit["BB_MONTH_COUNT"]
    )

    bb_credit["BB_DPD_30PLUS_RATIO"] = (
        bb_credit["BB_DPD_30PLUS_COUNT"] /
        bb_credit["BB_MONTH_COUNT"]
    )

    bb_credit["BB_DPD_90PLUS_RATIO"] = (
        bb_credit["BB_DPD_90PLUS_COUNT"] /
        bb_credit["BB_MONTH_COUNT"]
    )


    # ========================================================
    # 4. MAP EACH BUREAU CREDIT TO THE CUSTOMER
    # ========================================================

    bureau_id_map = (
        bureau_df[
            [
                "SK_ID_BUREAU",
                "SK_ID_CURR"
            ]
        ]
        .drop_duplicates()
    )


    bb_credit = bb_credit.merge(
        bureau_id_map,
        on="SK_ID_BUREAU",
        how="inner",
        validate="one_to_one"
    )


    # ========================================================
    # 5. SECOND LEVEL:
    # BUREAU CREDIT -> CUSTOMER
    # ========================================================

    bb_customer = (
        bb_credit
        .groupby("SK_ID_CURR")
        .agg(

            # Number of bureau credits with monthly history
            BB_CREDIT_COUNT=(
                "SK_ID_BUREAU",
                "count"
            ),

            # Total number of observed months
            BB_TOTAL_MONTHS=(
                "BB_MONTH_COUNT",
                "sum"
            ),

            # Longest available credit history
            BB_MAX_HISTORY_LENGTH=(
                "BB_HISTORY_LENGTH",
                "max"
            ),

            # Total delinquency months across all credits
            BB_TOTAL_DPD_MONTHS=(
                "BB_DPD_ANY_COUNT",
                "sum"
            ),

            BB_TOTAL_30PLUS_MONTHS=(
                "BB_DPD_30PLUS_COUNT",
                "sum"
            ),

            BB_TOTAL_90PLUS_MONTHS=(
                "BB_DPD_90PLUS_COUNT",
                "sum"
            ),

            BB_TOTAL_120PLUS_MONTHS=(
                "BB_DPD_120PLUS_COUNT",
                "sum"
            ),

            # Worst delinquency observed
            BB_MAX_SEVERITY=(
                "BB_MAX_SEVERITY",
                "max"
            ),

            # Average delinquency behavior across credits
            BB_MEAN_DPD_RATIO=(
                "BB_DPD_ANY_RATIO",
                "mean"
            ),

            BB_MEAN_30PLUS_RATIO=(
                "BB_DPD_30PLUS_RATIO",
                "mean"
            ),

            BB_MEAN_90PLUS_RATIO=(
                "BB_DPD_90PLUS_RATIO",
                "mean"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 6. CUSTOMER-LEVEL EVER FLAGS
    # ========================================================

    bb_customer["BB_EVER_DPD"] = (
        bb_customer["BB_TOTAL_DPD_MONTHS"] > 0
    ).astype("int8")


    bb_customer["BB_EVER_30PLUS"] = (
        bb_customer["BB_TOTAL_30PLUS_MONTHS"] > 0
    ).astype("int8")


    bb_customer["BB_EVER_90PLUS"] = (
        bb_customer["BB_TOTAL_90PLUS_MONTHS"] > 0
    ).astype("int8")


    bb_customer["BB_EVER_120PLUS"] = (
        bb_customer["BB_TOTAL_120PLUS_MONTHS"] > 0
    ).astype("int8")


    return bb_customer

In [ ]:
# ============================================================
# CREATE BUREAU BALANCE BASELINE FEATURES
# ============================================================

bb_features = build_bureau_balance_features(
    bureau_balance,
    bureau
)


print("=" * 70)
print("BUREAU BALANCE FEATURES")
print("=" * 70)

print(
    "Unique customers:",
    bb_features["SK_ID_CURR"].nunique()
)

print(
    "Number of BB features:",
    bb_features.shape[1] - 1
)

print(
    "Shape:",
    bb_features.shape
)

display(
    bb_features.head()
)

In [ ]:
# ============================================================
# MERGE BUREAU BALANCE FEATURES
# ============================================================

print("=" * 70)
print("MERGING BUREAU BALANCE FEATURES")
print("=" * 70)

print("Before merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    bb_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


X_test_base = X_test_base.merge(
    bb_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


print("\nRows preserved:")
print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test :",
    len(X_test_base) == len(application_test)
)

## 7. Previous-application history

Previous-application features summarize the customer's earlier Home Credit applications and approval/refusal history.

In [ ]:
# ============================================================
# BASELINE — PREVIOUS APPLICATION FEATURES
# Reproduction of original V7
# ============================================================

def build_previous_application_features(prev_df):

    prev = prev_df.copy()

    # --------------------------------------------------------
    # Helper for sums:
    # keep fully-missing groups as NaN instead of 0
    # --------------------------------------------------------
    def sum_with_nan(x):
        return x.sum(min_count=1)


    # ========================================================
    # 1. ROW-LEVEL STATUS FEATURES
    # ========================================================

    prev["IS_APPROVED"] = (
        prev["NAME_CONTRACT_STATUS"] == "Approved"
    ).astype("int8")

    prev["IS_REFUSED"] = (
        prev["NAME_CONTRACT_STATUS"] == "Refused"
    ).astype("int8")

    prev["IS_CANCELED"] = (
        prev["NAME_CONTRACT_STATUS"] == "Canceled"
    ).astype("int8")

    prev["IS_UNUSED"] = (
        prev["NAME_CONTRACT_STATUS"] == "Unused offer"
    ).astype("int8")


    # ========================================================
    # 2. APPLICATION AGE
    # ========================================================
    # DAYS_DECISION is negative because it happened
    # before the current application.
    #
    # Example:
    # -100  -> 100 days ago
    # -1000 -> 1000 days ago
    # ========================================================

    prev["PREV_DECISION_AGE"] = (
        -prev["DAYS_DECISION"]
    )


    # ========================================================
    # 3. GRANTED CREDIT / REQUESTED AMOUNT
    # ========================================================

    prev["PREV_CREDIT_APPLICATION_RATIO"] = (
        prev["AMT_CREDIT"] /
        prev["AMT_APPLICATION"]
    )

    prev["PREV_CREDIT_APPLICATION_RATIO"] = (
        prev["PREV_CREDIT_APPLICATION_RATIO"]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )


    # ========================================================
    # 4. CUSTOMER-LEVEL AGGREGATION
    # ========================================================

    prev_agg = (
        prev
        .groupby("SK_ID_CURR")
        .agg(

            # -----------------------------------------------
            # Number of previous applications
            # -----------------------------------------------

            PREV_APPLICATION_COUNT=(
                "SK_ID_PREV",
                "count"
            ),


            # -----------------------------------------------
            # Application outcomes
            # -----------------------------------------------

            PREV_APPROVED_COUNT=(
                "IS_APPROVED",
                "sum"
            ),

            PREV_REFUSED_COUNT=(
                "IS_REFUSED",
                "sum"
            ),

            PREV_CANCELED_COUNT=(
                "IS_CANCELED",
                "sum"
            ),

            PREV_UNUSED_COUNT=(
                "IS_UNUSED",
                "sum"
            ),


            # -----------------------------------------------
            # Amount originally requested
            # -----------------------------------------------

            PREV_APPLICATION_AMOUNT_MEAN=(
                "AMT_APPLICATION",
                "mean"
            ),

            PREV_APPLICATION_AMOUNT_MAX=(
                "AMT_APPLICATION",
                "max"
            ),


            # -----------------------------------------------
            # Amount actually granted
            # -----------------------------------------------

            PREV_CREDIT_MEAN=(
                "AMT_CREDIT",
                "mean"
            ),

            PREV_CREDIT_MAX=(
                "AMT_CREDIT",
                "max"
            ),

            PREV_CREDIT_TOTAL=(
                "AMT_CREDIT",
                sum_with_nan
            ),


            # -----------------------------------------------
            # Previous annuity burden
            # -----------------------------------------------

            PREV_ANNUITY_MEAN=(
                "AMT_ANNUITY",
                "mean"
            ),

            PREV_ANNUITY_MAX=(
                "AMT_ANNUITY",
                "max"
            ),


            # -----------------------------------------------
            # Down payment
            # -----------------------------------------------

            PREV_DOWN_PAYMENT_MEAN=(
                "AMT_DOWN_PAYMENT",
                "mean"
            ),


            # -----------------------------------------------
            # Previous contract payment duration
            # -----------------------------------------------

            PREV_PAYMENT_COUNT_MEAN=(
                "CNT_PAYMENT",
                "mean"
            ),

            PREV_PAYMENT_COUNT_MAX=(
                "CNT_PAYMENT",
                "max"
            ),


            # -----------------------------------------------
            # Previous-application recency
            # -----------------------------------------------

            PREV_RECENT_APPLICATION_DAYS=(
                "PREV_DECISION_AGE",
                "min"
            ),

            PREV_MEAN_APPLICATION_AGE=(
                "PREV_DECISION_AGE",
                "mean"
            ),

            PREV_OLDEST_APPLICATION_DAYS=(
                "PREV_DECISION_AGE",
                "max"
            ),


            # -----------------------------------------------
            # Granted/requested relationship
            # -----------------------------------------------

            PREV_CREDIT_APPLICATION_RATIO_MEAN=(
                "PREV_CREDIT_APPLICATION_RATIO",
                "mean"
            ),

            PREV_CREDIT_APPLICATION_RATIO_MAX=(
                "PREV_CREDIT_APPLICATION_RATIO",
                "max"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 5. CUSTOMER-LEVEL STATUS RATES
    # ========================================================

    prev_agg["PREV_APPROVAL_RATE"] = (
        prev_agg["PREV_APPROVED_COUNT"] /
        prev_agg["PREV_APPLICATION_COUNT"]
    )

    prev_agg["PREV_REFUSAL_RATE"] = (
        prev_agg["PREV_REFUSED_COUNT"] /
        prev_agg["PREV_APPLICATION_COUNT"]
    )

    prev_agg["PREV_CANCELED_RATE"] = (
        prev_agg["PREV_CANCELED_COUNT"] /
        prev_agg["PREV_APPLICATION_COUNT"]
    )


    # ========================================================
    # 6. CLEAN INFINITE VALUES
    # ========================================================

    prev_agg.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return prev_agg

In [ ]:
# ============================================================
# CREATE PREVIOUS-APPLICATION BASELINE FEATURES
# ============================================================

prev_features = build_previous_application_features(
    previous_application
)


print("=" * 70)
print("PREVIOUS APPLICATION FEATURES")
print("=" * 70)

print(
    "Original rows:",
    len(previous_application)
)

print(
    "Unique customers:",
    previous_application["SK_ID_CURR"].nunique()
)

print(
    "Aggregated customers:",
    len(prev_features)
)

print(
    "Number of features:",
    prev_features.shape[1] - 1
)

print(
    "Shape:",
    prev_features.shape
)

display(
    prev_features.head()
)

In [ ]:
# ============================================================
# MERGE PREVIOUS APPLICATION FEATURES
# ============================================================

print("=" * 70)
print("MERGING PREVIOUS APPLICATION FEATURES")
print("=" * 70)

print("Before merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    prev_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

X_test_base = X_test_base.merge(
    prev_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


print("\nRows preserved:")
print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test:",
    len(X_test_base) == len(application_test)
)

## 8. Installment-payment behavior

Installment features measure lateness, delay severity, underpayment, and payment behavior.

In [ ]:
# ============================================================
# BASELINE — INSTALLMENT PAYMENT FEATURES
# Reproduction of original V8
# ============================================================

def build_installment_features(installments_df):

    inst = installments_df.copy()

    # ========================================================
    # 1. ROW-LEVEL PAYMENT BEHAVIOR
    # ========================================================

    # Positive value = payment was late
    inst["PAYMENT_DELAY"] = (
        inst["DAYS_ENTRY_PAYMENT"] -
        inst["DAYS_INSTALMENT"]
    )

    # Positive value = customer paid less than scheduled
    inst["PAYMENT_DIFF"] = (
        inst["AMT_INSTALMENT"] -
        inst["AMT_PAYMENT"]
    )


    # --------------------------------------------------------
    # Late-payment flags
    # --------------------------------------------------------

    inst["LATE"] = (
        inst["PAYMENT_DELAY"] > 0
    ).astype("int8")

    inst["LATE_30"] = (
        inst["PAYMENT_DELAY"] > 30
    ).astype("int8")

    inst["LATE_60"] = (
        inst["PAYMENT_DELAY"] > 60
    ).astype("int8")

    inst["LATE_90"] = (
        inst["PAYMENT_DELAY"] > 90
    ).astype("int8")


    # Keep only positive lateness severity.
    # Early/on-time payments become 0.
    inst["LATE_DAYS"] = (
        inst["PAYMENT_DELAY"]
        .clip(lower=0)
    )


    # --------------------------------------------------------
    # Underpayment behavior
    # --------------------------------------------------------

    inst["UNDERPAYMENT"] = (
        inst["PAYMENT_DIFF"] > 0
    ).astype("int8")

    inst["UNDERPAYMENT_AMOUNT"] = (
        inst["PAYMENT_DIFF"]
        .clip(lower=0)
    )


    # --------------------------------------------------------
    # Actual payment relative to scheduled installment
    # --------------------------------------------------------

    inst["PAYMENT_RATIO"] = np.where(
        inst["AMT_INSTALMENT"] > 0,
        inst["AMT_PAYMENT"] /
        inst["AMT_INSTALMENT"],
        np.nan
    )


    # ========================================================
    # 2. AGGREGATE TO CUSTOMER LEVEL
    # ========================================================

    inst_agg = (
        inst
        .groupby("SK_ID_CURR")
        .agg(

            # -----------------------------------------------
            # History size
            # -----------------------------------------------

            INST_PAYMENT_COUNT=(
                "SK_ID_PREV",
                "size"
            ),

            INST_PREV_LOAN_COUNT=(
                "SK_ID_PREV",
                "nunique"
            ),


            # -----------------------------------------------
            # Delay behavior
            # -----------------------------------------------

            INST_LATE_COUNT=(
                "LATE",
                "sum"
            ),

            INST_LATE_30_COUNT=(
                "LATE_30",
                "sum"
            ),

            INST_LATE_60_COUNT=(
                "LATE_60",
                "sum"
            ),

            INST_LATE_90_COUNT=(
                "LATE_90",
                "sum"
            ),

            INST_MEAN_DELAY=(
                "PAYMENT_DELAY",
                "mean"
            ),

            INST_MAX_DELAY=(
                "PAYMENT_DELAY",
                "max"
            ),

            INST_MEAN_LATE_DAYS=(
                "LATE_DAYS",
                "mean"
            ),

            INST_MAX_LATE_DAYS=(
                "LATE_DAYS",
                "max"
            ),


            # -----------------------------------------------
            # Scheduled / actual amounts
            # -----------------------------------------------

            INST_TOTAL_SCHEDULED=(
                "AMT_INSTALMENT",
                "sum"
            ),

            INST_TOTAL_PAID=(
                "AMT_PAYMENT",
                "sum"
            ),

            INST_MEAN_SCHEDULED=(
                "AMT_INSTALMENT",
                "mean"
            ),

            INST_MEAN_PAID=(
                "AMT_PAYMENT",
                "mean"
            ),


            # -----------------------------------------------
            # Underpayment behavior
            # -----------------------------------------------

            INST_UNDERPAYMENT_COUNT=(
                "UNDERPAYMENT",
                "sum"
            ),

            INST_TOTAL_UNDERPAYMENT=(
                "UNDERPAYMENT_AMOUNT",
                "sum"
            ),

            INST_MEAN_PAYMENT_RATIO=(
                "PAYMENT_RATIO",
                "mean"
            ),

            INST_MIN_PAYMENT_RATIO=(
                "PAYMENT_RATIO",
                "min"
            ),


            # -----------------------------------------------
            # Historical timing
            # -----------------------------------------------

            INST_RECENT_PAYMENT_DAYS=(
                "DAYS_ENTRY_PAYMENT",
                "max"
            ),

            INST_OLDEST_PAYMENT_DAYS=(
                "DAYS_ENTRY_PAYMENT",
                "min"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 3. CUSTOMER-LEVEL RATIOS
    # ========================================================

    inst_agg["INST_LATE_RATIO"] = (
        inst_agg["INST_LATE_COUNT"] /
        inst_agg["INST_PAYMENT_COUNT"]
    )

    inst_agg["INST_LATE_30_RATIO"] = (
        inst_agg["INST_LATE_30_COUNT"] /
        inst_agg["INST_PAYMENT_COUNT"]
    )

    inst_agg["INST_LATE_60_RATIO"] = (
        inst_agg["INST_LATE_60_COUNT"] /
        inst_agg["INST_PAYMENT_COUNT"]
    )

    inst_agg["INST_LATE_90_RATIO"] = (
        inst_agg["INST_LATE_90_COUNT"] /
        inst_agg["INST_PAYMENT_COUNT"]
    )

    inst_agg["INST_UNDERPAYMENT_RATIO"] = (
        inst_agg["INST_UNDERPAYMENT_COUNT"] /
        inst_agg["INST_PAYMENT_COUNT"]
    )


    inst_agg["INST_TOTAL_PAYMENT_RATIO"] = np.where(
        inst_agg["INST_TOTAL_SCHEDULED"] > 0,

        inst_agg["INST_TOTAL_PAID"] /
        inst_agg["INST_TOTAL_SCHEDULED"],

        np.nan
    )


    # ========================================================
    # 4. EVER-BAD FLAGS
    # ========================================================

    inst_agg["INST_EVER_LATE"] = (
        inst_agg["INST_LATE_COUNT"] > 0
    ).astype("int8")

    inst_agg["INST_EVER_30_LATE"] = (
        inst_agg["INST_LATE_30_COUNT"] > 0
    ).astype("int8")

    inst_agg["INST_EVER_60_LATE"] = (
        inst_agg["INST_LATE_60_COUNT"] > 0
    ).astype("int8")

    inst_agg["INST_EVER_90_LATE"] = (
        inst_agg["INST_LATE_90_COUNT"] > 0
    ).astype("int8")

    inst_agg["INST_EVER_UNDERPAID"] = (
        inst_agg["INST_UNDERPAYMENT_COUNT"] > 0
    ).astype("int8")


    # ========================================================
    # 5. CONVERT TIMING INTO POSITIVE "AGE"
    # ========================================================

    inst_agg["INST_RECENT_PAYMENT_AGE"] = (
        -inst_agg["INST_RECENT_PAYMENT_DAYS"]
    )

    inst_agg["INST_OLDEST_PAYMENT_AGE"] = (
        -inst_agg["INST_OLDEST_PAYMENT_DAYS"]
    )


    # ========================================================
    # 6. CLEAN
    # ========================================================

    inst_agg.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return inst_agg

In [ ]:
# ============================================================
# CREATE INSTALLMENT BASELINE FEATURES
# ============================================================

inst_features = build_installment_features(
    installments
)


print("=" * 70)
print("INSTALLMENT PAYMENT FEATURES")
print("=" * 70)

print(
    "Original rows:",
    len(installments)
)

print(
    "Unique customers:",
    installments["SK_ID_CURR"].nunique()
)

print(
    "Aggregated customers:",
    len(inst_features)
)

print(
    "Number of features:",
    inst_features.shape[1] - 1
)

print(
    "Shape:",
    inst_features.shape
)

display(
    inst_features.head()
)

In [ ]:
# ============================================================
# MERGE INSTALLMENT FEATURES
# ============================================================

print("=" * 70)
print("MERGING INSTALLMENT FEATURES")
print("=" * 70)

print("Before merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    inst_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

X_test_base = X_test_base.merge(
    inst_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)

print("\nRows preserved:")

print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test:",
    len(X_test_base) == len(application_test)
)

## 9. POS_CASH history

POS_CASH features capture DPD behavior, remaining installments, and contract progression.

In [ ]:
# ============================================================
# BASELINE — POS_CASH FEATURES
# Reproduction of original V9
# ============================================================

def build_pos_cash_features(pos_df):

    pos = pos_df.copy()

    # ========================================================
    # 1. ROW-LEVEL BEHAVIOR
    # ========================================================

    # Any delinquency
    pos["POS_IS_DPD"] = (
        pos["SK_DPD"] > 0
    ).astype("int8")

    # Increasing delinquency severity
    pos["POS_IS_DPD_30"] = (
        pos["SK_DPD"] >= 30
    ).astype("int8")

    pos["POS_IS_DPD_60"] = (
        pos["SK_DPD"] >= 60
    ).astype("int8")

    pos["POS_IS_DPD_90"] = (
        pos["SK_DPD"] >= 90
    ).astype("int8")

    # DPD after tolerance
    pos["POS_IS_DPD_DEF"] = (
        pos["SK_DPD_DEF"] > 0
    ).astype("int8")


    # --------------------------------------------------------
    # Contract status
    # --------------------------------------------------------

    pos["POS_COMPLETED"] = (
        pos["NAME_CONTRACT_STATUS"] == "Completed"
    ).astype("int8")

    pos["POS_ACTIVE"] = (
        pos["NAME_CONTRACT_STATUS"] == "Active"
    ).astype("int8")

    pos["POS_DEMAND"] = (
        pos["NAME_CONTRACT_STATUS"] == "Demand"
    ).astype("int8")


    # ========================================================
    # 2. LOAN PROGRESS
    # ========================================================

    # Fraction of installments still remaining
    pos["POS_FUTURE_RATIO"] = (
        pos["CNT_INSTALMENT_FUTURE"] /
        pos["CNT_INSTALMENT"].replace(0, np.nan)
    )

    # Approximate fraction already completed
    pos["POS_COMPLETION_RATIO"] = (
        1 - pos["POS_FUTURE_RATIO"]
    )


    # ========================================================
    # 3. CUSTOMER-LEVEL AGGREGATION
    # ========================================================

    pos_agg = (
        pos
        .groupby("SK_ID_CURR")
        .agg(

            # -----------------------------------------------
            # History size
            # -----------------------------------------------

            POS_MONTH_COUNT=(
                "MONTHS_BALANCE",
                "count"
            ),

            POS_LOAN_COUNT=(
                "SK_ID_PREV",
                "nunique"
            ),


            # -----------------------------------------------
            # Timeline
            # -----------------------------------------------

            POS_RECENT_MONTH=(
                "MONTHS_BALANCE",
                "max"
            ),

            POS_OLDEST_MONTH=(
                "MONTHS_BALANCE",
                "min"
            ),


            # -----------------------------------------------
            # Installment structure
            # -----------------------------------------------

            POS_INSTALMENT_MEAN=(
                "CNT_INSTALMENT",
                "mean"
            ),

            POS_INSTALMENT_MAX=(
                "CNT_INSTALMENT",
                "max"
            ),

            POS_FUTURE_MEAN=(
                "CNT_INSTALMENT_FUTURE",
                "mean"
            ),

            POS_FUTURE_MIN=(
                "CNT_INSTALMENT_FUTURE",
                "min"
            ),

            POS_FUTURE_RATIO_MEAN=(
                "POS_FUTURE_RATIO",
                "mean"
            ),

            POS_COMPLETION_RATIO_MEAN=(
                "POS_COMPLETION_RATIO",
                "mean"
            ),


            # -----------------------------------------------
            # DPD severity
            # -----------------------------------------------

            POS_DPD_MEAN=(
                "SK_DPD",
                "mean"
            ),

            POS_DPD_MAX=(
                "SK_DPD",
                "max"
            ),

            POS_DPD_DEF_MEAN=(
                "SK_DPD_DEF",
                "mean"
            ),

            POS_DPD_DEF_MAX=(
                "SK_DPD_DEF",
                "max"
            ),


            # -----------------------------------------------
            # DPD frequency
            # -----------------------------------------------

            POS_DPD_COUNT=(
                "POS_IS_DPD",
                "sum"
            ),

            POS_DPD_30_COUNT=(
                "POS_IS_DPD_30",
                "sum"
            ),

            POS_DPD_60_COUNT=(
                "POS_IS_DPD_60",
                "sum"
            ),

            POS_DPD_90_COUNT=(
                "POS_IS_DPD_90",
                "sum"
            ),

            POS_DPD_DEF_COUNT=(
                "POS_IS_DPD_DEF",
                "sum"
            ),


            # -----------------------------------------------
            # Contract status
            # -----------------------------------------------

            POS_COMPLETED_COUNT=(
                "POS_COMPLETED",
                "sum"
            ),

            POS_ACTIVE_COUNT=(
                "POS_ACTIVE",
                "sum"
            ),

            POS_DEMAND_COUNT=(
                "POS_DEMAND",
                "sum"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 4. NORMALIZED BEHAVIOR RATIOS
    # ========================================================

    denom = (
        pos_agg["POS_MONTH_COUNT"]
        .replace(0, np.nan)
    )

    pos_agg["POS_DPD_RATIO"] = (
        pos_agg["POS_DPD_COUNT"] /
        denom
    )

    pos_agg["POS_DPD_30_RATIO"] = (
        pos_agg["POS_DPD_30_COUNT"] /
        denom
    )

    pos_agg["POS_DPD_60_RATIO"] = (
        pos_agg["POS_DPD_60_COUNT"] /
        denom
    )

    pos_agg["POS_DPD_90_RATIO"] = (
        pos_agg["POS_DPD_90_COUNT"] /
        denom
    )

    pos_agg["POS_DPD_DEF_RATIO"] = (
        pos_agg["POS_DPD_DEF_COUNT"] /
        denom
    )

    pos_agg["POS_COMPLETED_RATIO"] = (
        pos_agg["POS_COMPLETED_COUNT"] /
        denom
    )

    pos_agg["POS_ACTIVE_RATIO"] = (
        pos_agg["POS_ACTIVE_COUNT"] /
        denom
    )


    # ========================================================
    # 5. EVER-BAD FLAGS
    # ========================================================

    pos_agg["POS_EVER_DPD"] = (
        pos_agg["POS_DPD_COUNT"] > 0
    ).astype("int8")

    pos_agg["POS_EVER_30_DPD"] = (
        pos_agg["POS_DPD_30_COUNT"] > 0
    ).astype("int8")

    pos_agg["POS_EVER_60_DPD"] = (
        pos_agg["POS_DPD_60_COUNT"] > 0
    ).astype("int8")

    pos_agg["POS_EVER_90_DPD"] = (
        pos_agg["POS_DPD_90_COUNT"] > 0
    ).astype("int8")

    pos_agg["POS_EVER_DPD_DEF"] = (
        pos_agg["POS_DPD_DEF_COUNT"] > 0
    ).astype("int8")


    # ========================================================
    # 6. HISTORY AGE
    # ========================================================

    # MONTHS_BALANCE:
    # -1  = one month ago
    # -20 = twenty months ago

    pos_agg["POS_RECENT_HISTORY_AGE"] = (
        -pos_agg["POS_RECENT_MONTH"]
    )

    pos_agg["POS_HISTORY_LENGTH"] = (
        pos_agg["POS_RECENT_MONTH"] -
        pos_agg["POS_OLDEST_MONTH"] +
        1
    )


    # ========================================================
    # CLEAN
    # ========================================================

    pos_agg.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return pos_agg

In [ ]:
# ============================================================
# CREATE POS_CASH BASELINE FEATURES
# ============================================================

pos_features = build_pos_cash_features(
    pos_cash
)

print("=" * 70)
print("POS_CASH FEATURES")
print("=" * 70)

print("Original rows:", len(pos_cash))

print(
    "Unique customers:",
    pos_cash["SK_ID_CURR"].nunique()
)

print(
    "Aggregated customers:",
    len(pos_features)
)

print(
    "Number of features:",
    pos_features.shape[1] - 1
)

print(
    "Shape:",
    pos_features.shape
)

display(
    pos_features.head()
)

In [ ]:
# ============================================================
# MERGE POS_CASH FEATURES
# ============================================================

print("=" * 70)
print("MERGING POS_CASH FEATURES")
print("=" * 70)

print("Before merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    pos_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

X_test_base = X_test_base.merge(
    pos_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)

print("\nRows preserved:")

print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test:",
    len(X_test_base) == len(application_test)
)

## 10. Credit-card history

Credit-card features capture utilization, balances, drawings, ATM behavior, minimum-payment behavior, and delinquency.

In [ ]:
# ============================================================
# BASELINE — CREDIT CARD FEATURES
# Reproduction of original V10
# ============================================================

def build_credit_card_features(cc_df):

    cc = cc_df.copy()

    # ========================================================
    # 1. ROW-LEVEL FEATURES
    # ========================================================

    # --------------------------------------------------------
    # Credit utilization
    # balance / credit limit
    # --------------------------------------------------------

    cc["CC_UTILIZATION"] = (
        cc["AMT_BALANCE"] /
        cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
    )

    cc["CC_HIGH_UTILIZATION"] = (
        cc["CC_UTILIZATION"] >= 0.80
    ).astype("int8")

    cc["CC_FULL_UTILIZATION"] = (
        cc["CC_UTILIZATION"] >= 1.00
    ).astype("int8")


    # --------------------------------------------------------
    # Payment behavior
    # --------------------------------------------------------

    cc["CC_PAYMENT_MIN_RATIO"] = (
        cc["AMT_PAYMENT_CURRENT"] /
        cc["AMT_INST_MIN_REGULARITY"].replace(0, np.nan)
    )

    cc["CC_BELOW_MIN_PAYMENT"] = (
        (cc["AMT_INST_MIN_REGULARITY"] > 0)
        &
        (
            cc["AMT_PAYMENT_CURRENT"] <
            cc["AMT_INST_MIN_REGULARITY"]
        )
    ).astype("int8")


    # --------------------------------------------------------
    # Drawing behavior
    # --------------------------------------------------------

    cc["CC_HAS_DRAWING"] = (
        cc["AMT_DRAWINGS_CURRENT"] > 0
    ).astype("int8")

    cc["CC_HAS_ATM_DRAWING"] = (
        cc["AMT_DRAWINGS_ATM_CURRENT"] > 0
    ).astype("int8")


    # --------------------------------------------------------
    # Delinquency
    # --------------------------------------------------------

    cc["CC_DPD"] = (
        cc["SK_DPD"] > 0
    ).astype("int8")

    cc["CC_DPD_30"] = (
        cc["SK_DPD"] >= 30
    ).astype("int8")

    cc["CC_DPD_60"] = (
        cc["SK_DPD"] >= 60
    ).astype("int8")

    cc["CC_DPD_90"] = (
        cc["SK_DPD"] >= 90
    ).astype("int8")

    cc["CC_DPD_DEF"] = (
        cc["SK_DPD_DEF"] > 0
    ).astype("int8")


    # ========================================================
    # 2. CUSTOMER-LEVEL AGGREGATION
    # ========================================================

    cc_customer = (
        cc
        .groupby("SK_ID_CURR")
        .agg(

            # -----------------------------------------------
            # History / exposure
            # -----------------------------------------------

            CC_MONTH_COUNT=(
                "MONTHS_BALANCE",
                "count"
            ),

            CC_CARD_COUNT=(
                "SK_ID_PREV",
                "nunique"
            ),

            CC_RECENT_MONTH=(
                "MONTHS_BALANCE",
                "max"
            ),

            CC_OLDEST_MONTH=(
                "MONTHS_BALANCE",
                "min"
            ),


            # -----------------------------------------------
            # Balance
            # -----------------------------------------------

            CC_BALANCE_MEAN=(
                "AMT_BALANCE",
                "mean"
            ),

            CC_BALANCE_MAX=(
                "AMT_BALANCE",
                "max"
            ),

            CC_BALANCE_SUM=(
                "AMT_BALANCE",
                "sum"
            ),


            # -----------------------------------------------
            # Credit limit
            # -----------------------------------------------

            CC_LIMIT_MEAN=(
                "AMT_CREDIT_LIMIT_ACTUAL",
                "mean"
            ),

            CC_LIMIT_MAX=(
                "AMT_CREDIT_LIMIT_ACTUAL",
                "max"
            ),


            # -----------------------------------------------
            # Utilization
            # -----------------------------------------------

            CC_UTILIZATION_MEAN=(
                "CC_UTILIZATION",
                "mean"
            ),

            CC_UTILIZATION_MAX=(
                "CC_UTILIZATION",
                "max"
            ),

            CC_HIGH_UTILIZATION_RATIO=(
                "CC_HIGH_UTILIZATION",
                "mean"
            ),

            CC_FULL_UTILIZATION_RATIO=(
                "CC_FULL_UTILIZATION",
                "mean"
            ),


            # -----------------------------------------------
            # Drawings
            # -----------------------------------------------

            CC_DRAWINGS_MEAN=(
                "AMT_DRAWINGS_CURRENT",
                "mean"
            ),

            CC_DRAWINGS_MAX=(
                "AMT_DRAWINGS_CURRENT",
                "max"
            ),

            CC_DRAWINGS_TOTAL=(
                "AMT_DRAWINGS_CURRENT",
                "sum"
            ),

            CC_DRAWING_COUNT_MEAN=(
                "CNT_DRAWINGS_CURRENT",
                "mean"
            ),

            CC_DRAWING_MONTH_RATIO=(
                "CC_HAS_DRAWING",
                "mean"
            ),

            CC_ATM_DRAWING_MONTH_RATIO=(
                "CC_HAS_ATM_DRAWING",
                "mean"
            ),


            # -----------------------------------------------
            # Payments
            # -----------------------------------------------

            CC_PAYMENT_MEAN=(
                "AMT_PAYMENT_CURRENT",
                "mean"
            ),

            CC_PAYMENT_TOTAL=(
                "AMT_PAYMENT_TOTAL_CURRENT",
                "sum"
            ),

            CC_MIN_PAYMENT_MEAN=(
                "AMT_INST_MIN_REGULARITY",
                "mean"
            ),

            CC_PAYMENT_MIN_RATIO_MEAN=(
                "CC_PAYMENT_MIN_RATIO",
                "mean"
            ),

            CC_PAYMENT_MIN_RATIO_MIN=(
                "CC_PAYMENT_MIN_RATIO",
                "min"
            ),

            CC_BELOW_MIN_PAYMENT_RATIO=(
                "CC_BELOW_MIN_PAYMENT",
                "mean"
            ),


            # -----------------------------------------------
            # Receivable
            # -----------------------------------------------

            CC_RECEIVABLE_MEAN=(
                "AMT_TOTAL_RECEIVABLE",
                "mean"
            ),

            CC_RECEIVABLE_MAX=(
                "AMT_TOTAL_RECEIVABLE",
                "max"
            ),


            # -----------------------------------------------
            # Delinquency severity
            # -----------------------------------------------

            CC_DPD_MEAN=(
                "SK_DPD",
                "mean"
            ),

            CC_DPD_MAX=(
                "SK_DPD",
                "max"
            ),

            CC_DPD_DEF_MEAN=(
                "SK_DPD_DEF",
                "mean"
            ),

            CC_DPD_DEF_MAX=(
                "SK_DPD_DEF",
                "max"
            ),


            # -----------------------------------------------
            # Delinquency frequency
            # -----------------------------------------------

            CC_DPD_RATIO=(
                "CC_DPD",
                "mean"
            ),

            CC_DPD_30_RATIO=(
                "CC_DPD_30",
                "mean"
            ),

            CC_DPD_60_RATIO=(
                "CC_DPD_60",
                "mean"
            ),

            CC_DPD_90_RATIO=(
                "CC_DPD_90",
                "mean"
            ),

            CC_DPD_DEF_RATIO=(
                "CC_DPD_DEF",
                "mean"
            ),


            # -----------------------------------------------
            # Ever delinquent
            # -----------------------------------------------

            CC_EVER_DPD=(
                "CC_DPD",
                "max"
            ),

            CC_EVER_30_DPD=(
                "CC_DPD_30",
                "max"
            ),

            CC_EVER_60_DPD=(
                "CC_DPD_60",
                "max"
            ),

            CC_EVER_90_DPD=(
                "CC_DPD_90",
                "max"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 3. HISTORY AGE
    # ========================================================

    cc_customer["CC_RECENT_HISTORY_AGE"] = (
        -cc_customer["CC_RECENT_MONTH"]
    )

    cc_customer["CC_HISTORY_LENGTH"] = (
        cc_customer["CC_RECENT_MONTH"] -
        cc_customer["CC_OLDEST_MONTH"] +
        1
    )


    # ========================================================
    # 4. CLEAN EXTREME RATIOS
    # ========================================================

    # Preserve NaNs.
    # Only cap pathological valid values.

    cc_customer["CC_PAYMENT_MIN_RATIO_MEAN"] = (
        cc_customer["CC_PAYMENT_MIN_RATIO_MEAN"]
        .clip(upper=20)
    )

    cc_customer["CC_PAYMENT_MIN_RATIO_MIN"] = (
        cc_customer["CC_PAYMENT_MIN_RATIO_MIN"]
        .clip(upper=20)
    )

    # Utilization > 1 can be meaningful:
    # balance can exceed the nominal limit.
    cc_customer["CC_UTILIZATION_MEAN"] = (
        cc_customer["CC_UTILIZATION_MEAN"]
        .clip(lower=-2, upper=5)
    )

    cc_customer["CC_UTILIZATION_MAX"] = (
        cc_customer["CC_UTILIZATION_MAX"]
        .clip(lower=-2, upper=10)
    )


    cc_customer.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return cc_customer

In [ ]:
# ============================================================
# CREATE CREDIT CARD BASELINE FEATURES
# ============================================================

cc_features = build_credit_card_features(
    credit_card
)


print("=" * 70)
print("CREDIT CARD FEATURES")
print("=" * 70)

print(
    "Original rows:",
    len(credit_card)
)

print(
    "Unique customers:",
    credit_card["SK_ID_CURR"].nunique()
)

print(
    "Aggregated customers:",
    len(cc_features)
)

print(
    "Number of features:",
    cc_features.shape[1] - 1
)

print(
    "Shape:",
    cc_features.shape
)

display(
    cc_features.head()
)

In [ ]:
# ============================================================
# MERGE CREDIT CARD FEATURES
# ============================================================

print("=" * 70)
print("MERGING CREDIT CARD FEATURES")
print("=" * 70)

print("Before merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    cc_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

X_test_base = X_test_base.merge(
    cc_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter merge:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)

print("\nRows preserved:")

print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test:",
    len(X_test_base) == len(application_test)
)

## 11. Recent behavioral windows

Fixed 6/12/24-month windows explicitly model recency. This was one of the strongest later-stage improvements.

In [ ]:
# ============================================================
# BASELINE — V11 RECENT BEHAVIOR
# Installments + Credit Card
# ============================================================

def build_recent_behavior_features(
    installments_df,
    credit_card_df
):

    # ========================================================
    # PART 1 — INSTALLMENT RECENCY
    # ========================================================

    inst = installments_df.copy()

    # Recreate row-level behavior used in V8
    inst["PAYMENT_DELAY"] = (
        inst["DAYS_ENTRY_PAYMENT"] -
        inst["DAYS_INSTALMENT"]
    )

    inst["PAYMENT_DIFF"] = (
        inst["AMT_INSTALMENT"] -
        inst["AMT_PAYMENT"]
    )

    inst["LATE"] = (
        inst["PAYMENT_DELAY"] > 0
    ).astype("int8")

    inst["LATE_30"] = (
        inst["PAYMENT_DELAY"] > 30
    ).astype("int8")

    inst["UNDERPAID"] = (
        inst["AMT_PAYMENT"] <
        inst["AMT_INSTALMENT"]
    ).astype("int8")


    # --------------------------------------------------------
    # Timeline safety
    # Only events BEFORE the current application
    # --------------------------------------------------------

    inst = inst[
        inst["DAYS_INSTALMENT"] < 0
    ].copy()


    # ========================================================
    # INSTALLMENT WINDOW FUNCTION
    # ========================================================

    def aggregate_inst_window(
        df,
        days,
        label
    ):

        temp = df[
            df["DAYS_INSTALMENT"] >= -days
        ]

        result = (
            temp
            .groupby("SK_ID_CURR")
            .agg(

                **{
                    f"INST_{label}_COUNT":
                        ("SK_ID_PREV", "size"),

                    f"INST_{label}_LOAN_COUNT":
                        ("SK_ID_PREV", "nunique"),

                    f"INST_{label}_LATE_RATIO":
                        ("LATE", "mean"),

                    f"INST_{label}_LATE30_RATIO":
                        ("LATE_30", "mean"),

                    f"INST_{label}_MEAN_DELAY":
                        ("PAYMENT_DELAY", "mean"),

                    f"INST_{label}_MAX_DELAY":
                        ("PAYMENT_DELAY", "max"),

                    f"INST_{label}_UNDERPAY_RATIO":
                        ("UNDERPAID", "mean"),

                    f"INST_{label}_MEAN_PAYMENT":
                        ("AMT_PAYMENT", "mean"),

                    f"INST_{label}_TOTAL_PAYMENT":
                        ("AMT_PAYMENT", "sum"),

                    f"INST_{label}_MEAN_SCHEDULED":
                        ("AMT_INSTALMENT", "mean"),

                    f"INST_{label}_TOTAL_SCHEDULED":
                        ("AMT_INSTALMENT", "sum")
                }
            )
            .reset_index()
        )

        return result


    # --------------------------------------------------------
    # Build 6 / 12 / 24 month installment windows
    # --------------------------------------------------------

    inst_6m = aggregate_inst_window(
        inst,
        180,
        "6M"
    )

    inst_12m = aggregate_inst_window(
        inst,
        365,
        "12M"
    )

    inst_24m = aggregate_inst_window(
        inst,
        730,
        "24M"
    )


    # ========================================================
    # PART 2 — CREDIT CARD RECENCY
    # ========================================================

    cc = credit_card_df.copy()


    # --------------------------------------------------------
    # Row-level V10 features
    # --------------------------------------------------------

    cc["CC_UTILIZATION"] = (
        cc["AMT_BALANCE"] /
        cc["AMT_CREDIT_LIMIT_ACTUAL"]
        .replace(0, np.nan)
    )

    cc["CC_HIGH_UTILIZATION"] = (
        cc["CC_UTILIZATION"] >= 0.80
    ).astype("int8")

    cc["CC_FULL_UTILIZATION"] = (
        cc["CC_UTILIZATION"] >= 1.00
    ).astype("int8")


    cc["CC_PAYMENT_MIN_RATIO"] = (
        cc["AMT_PAYMENT_CURRENT"] /
        cc["AMT_INST_MIN_REGULARITY"]
        .replace(0, np.nan)
    )


    cc["CC_BELOW_MIN_PAYMENT"] = (
        (cc["AMT_INST_MIN_REGULARITY"] > 0)
        &
        (
            cc["AMT_PAYMENT_CURRENT"] <
            cc["AMT_INST_MIN_REGULARITY"]
        )
    ).astype("int8")


    cc["CC_HAS_DRAWING"] = (
        cc["AMT_DRAWINGS_CURRENT"] > 0
    ).astype("int8")


    cc["CC_HAS_ATM_DRAWING"] = (
        cc["AMT_DRAWINGS_ATM_CURRENT"] > 0
    ).astype("int8")


    cc["CC_DPD"] = (
        cc["SK_DPD"] > 0
    ).astype("int8")

    cc["CC_DPD_30"] = (
        cc["SK_DPD"] >= 30
    ).astype("int8")

    cc["CC_DPD_60"] = (
        cc["SK_DPD"] >= 60
    ).astype("int8")

    cc["CC_DPD_90"] = (
        cc["SK_DPD"] >= 90
    ).astype("int8")

    cc["CC_DPD_DEF"] = (
        cc["SK_DPD_DEF"] > 0
    ).astype("int8")


    # --------------------------------------------------------
    # Only historical records
    # --------------------------------------------------------

    cc = cc[
        cc["MONTHS_BALANCE"] < 0
    ].copy()


    # ========================================================
    # CREDIT CARD WINDOW FUNCTION
    # ========================================================

    def aggregate_cc_window(
        df,
        months,
        label
    ):

        temp = df[
            df["MONTHS_BALANCE"] >= -months
        ]

        result = (
            temp
            .groupby("SK_ID_CURR")
            .agg(

                **{
                    f"CC_{label}_MONTH_COUNT":
                        ("SK_ID_PREV", "size"),

                    f"CC_{label}_CARD_COUNT":
                        ("SK_ID_PREV", "nunique"),


                    # Utilization
                    f"CC_{label}_UTIL_MEAN":
                        ("CC_UTILIZATION", "mean"),

                    f"CC_{label}_UTIL_MAX":
                        ("CC_UTILIZATION", "max"),

                    f"CC_{label}_HIGH_UTIL_RATIO":
                        ("CC_HIGH_UTILIZATION", "mean"),

                    f"CC_{label}_FULL_UTIL_RATIO":
                        ("CC_FULL_UTILIZATION", "mean"),


                    # Balance
                    f"CC_{label}_BALANCE_MEAN":
                        ("AMT_BALANCE", "mean"),

                    f"CC_{label}_BALANCE_MAX":
                        ("AMT_BALANCE", "max"),


                    # Drawing behavior
                    f"CC_{label}_DRAWING_RATIO":
                        ("CC_HAS_DRAWING", "mean"),

                    f"CC_{label}_ATM_DRAWING_RATIO":
                        ("CC_HAS_ATM_DRAWING", "mean"),

                    f"CC_{label}_DRAWINGS_MEAN":
                        ("AMT_DRAWINGS_CURRENT", "mean"),

                    f"CC_{label}_DRAWING_COUNT_MEAN":
                        ("CNT_DRAWINGS_CURRENT", "mean"),


                    # Payments
                    f"CC_{label}_PAYMENT_MEAN":
                        ("AMT_PAYMENT_CURRENT", "mean"),

                    f"CC_{label}_MIN_PAYMENT_MEAN":
                        ("AMT_INST_MIN_REGULARITY", "mean"),

                    f"CC_{label}_PAYMENT_MIN_RATIO":
                        ("CC_PAYMENT_MIN_RATIO", "mean"),

                    f"CC_{label}_BELOW_MIN_RATIO":
                        ("CC_BELOW_MIN_PAYMENT", "mean"),


                    # DPD
                    f"CC_{label}_DPD_RATIO":
                        ("CC_DPD", "mean"),

                    f"CC_{label}_DPD30_RATIO":
                        ("CC_DPD_30", "mean"),

                    f"CC_{label}_DPD60_RATIO":
                        ("CC_DPD_60", "mean"),

                    f"CC_{label}_DPD90_RATIO":
                        ("CC_DPD_90", "mean"),

                    f"CC_{label}_DPD_DEF_RATIO":
                        ("CC_DPD_DEF", "mean"),

                    f"CC_{label}_DPD_MEAN":
                        ("SK_DPD", "mean"),

                    f"CC_{label}_DPD_MAX":
                        ("SK_DPD", "max")
                }
            )
            .reset_index()
        )

        return result


    cc_6m = aggregate_cc_window(
        cc,
        6,
        "6M"
    )

    cc_12m = aggregate_cc_window(
        cc,
        12,
        "12M"
    )

    cc_24m = aggregate_cc_window(
        cc,
        24,
        "24M"
    )


    # ========================================================
    # PART 3 — MERGE ALL RECENCY TABLES
    # ========================================================

    recent_tables = [

        inst_6m,
        inst_12m,
        inst_24m,

        cc_6m,
        cc_12m,
        cc_24m
    ]


    recent = recent_tables[0]

    for table in recent_tables[1:]:

        recent = recent.merge(
            table,
            on="SK_ID_CURR",
            how="outer"
        )


    # ========================================================
    # PART 4 — INSTALLMENT TRENDS
    # ========================================================
    #
    # TREND =
    #
    # recent behavior
    #      -
    # longer-term behavior
    #
    # Example:
    #
    # recent late ratio = 0.30
    # 24M late ratio    = 0.10
    #
    # trend = +0.20
    #
    # Recent behavior is worse than historical behavior.
    # ========================================================

    recent["INST_LATE_TREND_6M_24M"] = (
        recent["INST_6M_LATE_RATIO"] -
        recent["INST_24M_LATE_RATIO"]
    )

    recent["INST_LATE30_TREND_6M_24M"] = (
        recent["INST_6M_LATE30_RATIO"] -
        recent["INST_24M_LATE30_RATIO"]
    )

    recent["INST_UNDERPAY_TREND_6M_24M"] = (
        recent["INST_6M_UNDERPAY_RATIO"] -
        recent["INST_24M_UNDERPAY_RATIO"]
    )

    recent["INST_DELAY_TREND_6M_24M"] = (
        recent["INST_6M_MEAN_DELAY"] -
        recent["INST_24M_MEAN_DELAY"]
    )


    # ========================================================
    # PART 5 — CREDIT CARD TRENDS
    # ========================================================

    recent["CC_UTIL_TREND_6M_24M"] = (
        recent["CC_6M_UTIL_MEAN"] -
        recent["CC_24M_UTIL_MEAN"]
    )

    recent["CC_HIGH_UTIL_TREND_6M_24M"] = (
        recent["CC_6M_HIGH_UTIL_RATIO"] -
        recent["CC_24M_HIGH_UTIL_RATIO"]
    )

    recent["CC_FULL_UTIL_TREND_6M_24M"] = (
        recent["CC_6M_FULL_UTIL_RATIO"] -
        recent["CC_24M_FULL_UTIL_RATIO"]
    )

    recent["CC_DRAWING_TREND_6M_24M"] = (
        recent["CC_6M_DRAWING_RATIO"] -
        recent["CC_24M_DRAWING_RATIO"]
    )

    recent["CC_ATM_TREND_6M_24M"] = (
        recent["CC_6M_ATM_DRAWING_RATIO"] -
        recent["CC_24M_ATM_DRAWING_RATIO"]
    )

    recent["CC_BELOW_MIN_TREND_6M_24M"] = (
        recent["CC_6M_BELOW_MIN_RATIO"] -
        recent["CC_24M_BELOW_MIN_RATIO"]
    )

    recent["CC_DPD_TREND_6M_24M"] = (
        recent["CC_6M_DPD_RATIO"] -
        recent["CC_24M_DPD_RATIO"]
    )


    # ========================================================
    # CLEAN
    # ========================================================

    recent.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return recent

In [ ]:
# ============================================================
# CREATE V11 RECENT FEATURES
# ============================================================

recent_features = build_recent_behavior_features(
    installments,
    credit_card
)


print("=" * 70)
print("V11 — RECENT BEHAVIOR FEATURES")
print("=" * 70)

print(
    "Unique customers:",
    recent_features["SK_ID_CURR"].nunique()
)

print(
    "Number of engineered features:",
    recent_features.shape[1] - 1
)

print(
    "Shape:",
    recent_features.shape
)

display(
    recent_features.head()
)

In [ ]:
# ============================================================
# MERGE V11 RECENT FEATURES
# ============================================================

print("=" * 70)
print("MERGING RECENT BEHAVIOR FEATURES")
print("=" * 70)

print("Before:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    recent_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

X_test_base = X_test_base.merge(
    recent_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


print("\nRows preserved:")

print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test:",
    len(X_test_base) == len(application_test)
)

## 12. Recent bureau windows

Recent bureau windows apply the same recency principle to external credit history.

In [ ]:
# ============================================================
# BASELINE — V12 RECENT BUREAU FEATURES
# ============================================================

def build_recent_bureau_features(bureau_df):

    bur = bureau_df.copy()

    # ========================================================
    # 1. ROW-LEVEL FEATURES
    # ========================================================

    # Positive age: 100 means credit opened 100 days ago
    bur["BUREAU_CREDIT_AGE"] = (
        -bur["DAYS_CREDIT"]
    )

    # Credit status
    bur["BUR_ACTIVE"] = (
        bur["CREDIT_ACTIVE"] == "Active"
    ).astype("int8")

    bur["BUR_CLOSED"] = (
        bur["CREDIT_ACTIVE"] == "Closed"
    ).astype("int8")

    # Debt presence
    bur["BUR_HAS_DEBT"] = (
        bur["AMT_CREDIT_SUM_DEBT"]
        .fillna(0) > 0
    ).astype("int8")

    # Overdue behavior
    bur["BUR_OVERDUE"] = (
        bur["CREDIT_DAY_OVERDUE"] > 0
    ).astype("int8")

    bur["BUR_HAS_OVERDUE_AMOUNT"] = (
        bur["AMT_CREDIT_SUM_OVERDUE"]
        .fillna(0) > 0
    ).astype("int8")

    # Individual-credit debt / credit ratio
    bur["BUR_DEBT_CREDIT_RATIO"] = (
        bur["AMT_CREDIT_SUM_DEBT"] /
        bur["AMT_CREDIT_SUM"].replace(0, np.nan)
    )

    bur["BUR_DEBT_CREDIT_RATIO"] = (
        bur["BUR_DEBT_CREDIT_RATIO"]
        .replace([np.inf, -np.inf], np.nan)
    )


    # ========================================================
    # 2. WINDOW AGGREGATION FUNCTION
    # ========================================================

    def aggregate_bureau_window(
        df,
        days,
        label
    ):

        # Credit must exist before current application
        # and be opened inside the requested lookback.
        temp = df[
            (df["DAYS_CREDIT"] <= 0)
            &
            (df["DAYS_CREDIT"] >= -days)
        ]

        result = (
            temp
            .groupby("SK_ID_CURR")
            .agg(

                **{
                    # ---------------------------------------
                    # History size
                    # ---------------------------------------

                    f"BUR_{label}_CREDIT_COUNT":
                        ("SK_ID_BUREAU", "count"),


                    # ---------------------------------------
                    # Active / closed
                    # ---------------------------------------

                    f"BUR_{label}_ACTIVE_COUNT":
                        ("BUR_ACTIVE", "sum"),

                    f"BUR_{label}_ACTIVE_RATIO":
                        ("BUR_ACTIVE", "mean"),

                    f"BUR_{label}_CLOSED_RATIO":
                        ("BUR_CLOSED", "mean"),


                    # ---------------------------------------
                    # Credit exposure
                    # ---------------------------------------

                    f"BUR_{label}_CREDIT_MEAN":
                        ("AMT_CREDIT_SUM", "mean"),

                    f"BUR_{label}_CREDIT_MAX":
                        ("AMT_CREDIT_SUM", "max"),

                    f"BUR_{label}_CREDIT_TOTAL":
                        ("AMT_CREDIT_SUM", "sum"),


                    # ---------------------------------------
                    # Debt
                    # ---------------------------------------

                    f"BUR_{label}_DEBT_MEAN":
                        ("AMT_CREDIT_SUM_DEBT", "mean"),

                    f"BUR_{label}_DEBT_MAX":
                        ("AMT_CREDIT_SUM_DEBT", "max"),

                    f"BUR_{label}_DEBT_TOTAL":
                        ("AMT_CREDIT_SUM_DEBT", "sum"),

                    f"BUR_{label}_HAS_DEBT_RATIO":
                        ("BUR_HAS_DEBT", "mean"),

                    f"BUR_{label}_DEBT_CREDIT_RATIO_MEAN":
                        ("BUR_DEBT_CREDIT_RATIO", "mean"),

                    f"BUR_{label}_DEBT_CREDIT_RATIO_MAX":
                        ("BUR_DEBT_CREDIT_RATIO", "max"),


                    # ---------------------------------------
                    # Overdue
                    # ---------------------------------------

                    f"BUR_{label}_OVERDUE_RATIO":
                        ("BUR_OVERDUE", "mean"),

                    f"BUR_{label}_MAX_OVERDUE_DAYS":
                        ("CREDIT_DAY_OVERDUE", "max"),

                    f"BUR_{label}_OVERDUE_AMOUNT_TOTAL":
                        ("AMT_CREDIT_SUM_OVERDUE", "sum"),

                    f"BUR_{label}_OVERDUE_AMOUNT_RATIO":
                        ("BUR_HAS_OVERDUE_AMOUNT", "mean"),


                    # ---------------------------------------
                    # Prolongation
                    # ---------------------------------------

                    f"BUR_{label}_PROLONG_TOTAL":
                        ("CNT_CREDIT_PROLONG", "sum"),

                    f"BUR_{label}_PROLONG_MEAN":
                        ("CNT_CREDIT_PROLONG", "mean"),


                    # ---------------------------------------
                    # Recency within the window
                    # ---------------------------------------

                    f"BUR_{label}_MEAN_CREDIT_AGE":
                        ("BUREAU_CREDIT_AGE", "mean"),

                    f"BUR_{label}_RECENT_CREDIT_AGE":
                        ("BUREAU_CREDIT_AGE", "min")
                }
            )
            .reset_index()
        )

        return result


    # ========================================================
    # 3. CREATE 6M / 12M / 24M WINDOWS
    # ========================================================

    bur_6m = aggregate_bureau_window(
        bur,
        180,
        "6M"
    )

    bur_12m = aggregate_bureau_window(
        bur,
        365,
        "12M"
    )

    bur_24m = aggregate_bureau_window(
        bur,
        730,
        "24M"
    )


    # ========================================================
    # 4. COMBINE WINDOWS
    # ========================================================

    recent_bur = bur_6m.merge(
        bur_12m,
        on="SK_ID_CURR",
        how="outer"
    )

    recent_bur = recent_bur.merge(
        bur_24m,
        on="SK_ID_CURR",
        how="outer"
    )


    # ========================================================
    # 5. TREND FEATURES
    # ========================================================

    # More recent active-credit pressure
    recent_bur["BUR_ACTIVE_TREND_6M_24M"] = (
        recent_bur["BUR_6M_ACTIVE_RATIO"] -
        recent_bur["BUR_24M_ACTIVE_RATIO"]
    )

    # Change in prevalence of outstanding debt
    recent_bur["BUR_DEBT_TREND_6M_24M"] = (
        recent_bur["BUR_6M_HAS_DEBT_RATIO"] -
        recent_bur["BUR_24M_HAS_DEBT_RATIO"]
    )

    # Change in relative debt burden
    recent_bur["BUR_DEBT_CREDIT_TREND_6M_24M"] = (
        recent_bur["BUR_6M_DEBT_CREDIT_RATIO_MEAN"] -
        recent_bur["BUR_24M_DEBT_CREDIT_RATIO_MEAN"]
    )

    # Change in overdue frequency
    recent_bur["BUR_OVERDUE_TREND_6M_24M"] = (
        recent_bur["BUR_6M_OVERDUE_RATIO"] -
        recent_bur["BUR_24M_OVERDUE_RATIO"]
    )

    # Change in typical credit size
    recent_bur["BUR_CREDIT_SIZE_TREND_6M_24M"] = (
        recent_bur["BUR_6M_CREDIT_MEAN"] -
        recent_bur["BUR_24M_CREDIT_MEAN"]
    )


    recent_bur.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

    return recent_bur

In [ ]:
# ============================================================
# CREATE V12 RECENT BUREAU FEATURES
# ============================================================

recent_bureau_features = build_recent_bureau_features(
    bureau
)


print("=" * 70)
print("V12 — RECENT BUREAU FEATURES")
print("=" * 70)

print(
    "Unique customers:",
    recent_bureau_features["SK_ID_CURR"].nunique()
)

print(
    "Number of engineered features:",
    recent_bureau_features.shape[1] - 1
)

print(
    "Shape:",
    recent_bureau_features.shape
)

display(
    recent_bureau_features.head()
)

In [ ]:
# ============================================================
# MERGE V12 RECENT BUREAU FEATURES
# ============================================================

print("=" * 70)
print("MERGING RECENT BUREAU FEATURES")
print("=" * 70)

print("Before:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


X_train_base = X_train_base.merge(
    recent_bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

X_test_base = X_test_base.merge(
    recent_bureau_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)


print("\nAfter:")
print("Train:", X_train_base.shape)
print("Test :", X_test_base.shape)


print("\nRows preserved:")

print(
    "Train:",
    len(X_train_base) == len(application_train)
)

print(
    "Test:",
    len(X_test_base) == len(application_test)
)

## 13. Robust bureau ratios

Extreme debt/credit ratios are stabilized with percentile clipping and signed-log transforms.

In [ ]:
# ============================================================
# BASELINE — V13 ROBUST BUREAU RATIOS
# ============================================================

def add_robust_bureau_ratios(
    train_df,
    test_df
):

    train_df = train_df.copy()
    test_df = test_df.copy()

    # --------------------------------------------------------
    # Select ONLY original recent-bureau debt/credit ratios.
    #
    # We exclude:
    # - trend features
    # - already-created robust features
    # --------------------------------------------------------

    ratio_cols = [
        col
        for col in train_df.columns

        if (
            col.startswith("BUR_")
            and "DEBT_CREDIT_RATIO" in col
            and "TREND" not in col
            and not col.endswith("_ROBUST_CLIP")
            and not col.endswith("_SIGNED_LOG")
        )
    ]


    print("=" * 75)
    print("V13 — ROBUST BUREAU RATIOS")
    print("=" * 75)

    print("\nRatio columns:")

    for col in ratio_cols:
        print(" ", col)

    print(
        "\nNumber of original ratios:",
        len(ratio_cols)
    )


    diagnostics = []


    # ========================================================
    # IMPORTANT:
    #
    # clipping limits are learned from TRAIN only.
    #
    # We then apply exactly the same limits to TEST.
    # ========================================================

    for col in ratio_cols:

        train_ratio = pd.to_numeric(
            train_df[col],
            errors="coerce"
        )

        test_ratio = pd.to_numeric(
            test_df[col],
            errors="coerce"
        )


        # ----------------------------------------------------
        # Clean infinity
        # ----------------------------------------------------

        train_ratio = train_ratio.replace(
            [np.inf, -np.inf],
            np.nan
        )

        test_ratio = test_ratio.replace(
            [np.inf, -np.inf],
            np.nan
        )


        # ----------------------------------------------------
        # Learn clipping thresholds from TRAIN
        #
        # 0.5 percentile
        # 99.5 percentile
        # ----------------------------------------------------

        q005 = train_ratio.quantile(0.005)
        q995 = train_ratio.quantile(0.995)


        # ====================================================
        # A. WINSORIZED / CLIPPED VERSION
        # ====================================================

        clip_col = (
            col +
            "_ROBUST_CLIP"
        )

        train_df[clip_col] = (
            train_ratio.clip(
                lower=q005,
                upper=q995
            )
        )

        test_df[clip_col] = (
            test_ratio.clip(
                lower=q005,
                upper=q995
            )
        )


        # ====================================================
        # B. SIGNED LOG VERSION
        # ====================================================
        #
        # log1p compresses large magnitude values.
        #
        # Example:
        #
        # 10000 -> ~9.21
        # 1000  -> ~6.91
        # 10    -> ~2.40
        # 1     -> ~0.69
        #
        # Ordinary log cannot represent negative values.
        #
        # signed log:
        #
        # sign(x) * log(1 + |x|)
        #
        # preserves the sign.
        # ====================================================

        log_col = (
            col +
            "_SIGNED_LOG"
        )

        train_df[log_col] = (
            np.sign(train_ratio)
            *
            np.log1p(
                np.abs(train_ratio)
            )
        )

        test_df[log_col] = (
            np.sign(test_ratio)
            *
            np.log1p(
                np.abs(test_ratio)
            )
        )


        diagnostics.append(
            {
                "Feature": col,
                "P0.5": q005,
                "P99.5": q995,

                "Train_Min":
                    train_ratio.min(),

                "Train_Max":
                    train_ratio.max(),

                "Clipped_Min":
                    train_df[clip_col].min(),

                "Clipped_Max":
                    train_df[clip_col].max()
            }
        )


    diagnostics = pd.DataFrame(
        diagnostics
    )


    print(
        "\nNew V13 features:",
        len(ratio_cols) * 2
    )


    return (
        train_df,
        test_df,
        diagnostics,
        ratio_cols
    )

In [ ]:
# ============================================================
# APPLY V13 ROBUST RATIOS
# ============================================================

X_train_base, X_test_base, robust_diagnostics, robust_ratio_cols = (
    add_robust_bureau_ratios(
        X_train_base,
        X_test_base
    )
)


print("\n" + "=" * 70)
print("AFTER V13")
print("=" * 70)

print(
    "Train shape:",
    X_train_base.shape
)

print(
    "Test shape :",
    X_test_base.shape
)

print(
    "Number of robust features:",
    len(robust_ratio_cols) * 2
)


display(
    robust_diagnostics
)

## 14. Frozen baseline validation

After reconstructing the strong V13 feature set, the baseline was validated with 5-fold stratified CV.

**Reproduction result**

| Metric | Original V13 | Reconstructed notebook |
|---|---:|---:|
| OOF ROC-AUC | 0.791549 | **0.791721** |
| Difference | — | **+0.000172** |

The difference is negligible relative to fold-to-fold variation, so **0.791721** is treated as the reproduced full-CV baseline.

For winner-inspired research, training cost became significant. Therefore the first three folds of the same fixed 5-fold partition are used as a screening protocol. The known 3-fold reference mean is **0.792002**. Promising ideas can later be promoted to full CV.

In [ ]:
# ============================================================
# REUSABLE LIGHTGBM VALIDATION
# ============================================================

BASELINE_FAST_SCORES = np.array([0.787794, 0.798407, 0.789805])
BASELINE_FAST_MEAN = BASELINE_FAST_SCORES.mean()

def prepare_model_matrix(df):
    X = df.copy()
    if "SK_ID_CURR" in X.columns:
        X = X.drop(columns=["SK_ID_CURR"])

    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in cat_cols:
        X[col] = X[col].astype("category")

    return X, cat_cols


def run_lgbm_cv(X_df, y, folds_to_run=3, verbose=True):
    X, cat_cols = prepare_model_matrix(X_df)

    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores, iterations = [], []
    start = time.time()

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        if fold > folds_to_run:
            break

        model = lgb.LGBMClassifier(
            objective="binary",
            metric="auc",
            n_estimators=5000,
            learning_rate=0.02,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=40,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=0.1,
            random_state=RANDOM_STATE + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
            eval_set=[(X.iloc[valid_idx], y.iloc[valid_idx])],
            eval_metric="auc",
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )

        pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
        auc = roc_auc_score(y.iloc[valid_idx], pred)

        scores.append(auc)
        iterations.append(model.best_iteration_)

        if verbose:
            print(f"Fold {fold}: {auc:.6f} | iteration {model.best_iteration_}")

    return {
        "scores": np.array(scores),
        "mean_auc": float(np.mean(scores)),
        "mean_iteration": float(np.mean(iterations)),
        "runtime_min": (time.time() - start) / 60,
    }

# Part II — Winner-Inspired Hypothesis Testing

The purpose of this phase is **not to copy a winning notebook**. Each technique is treated as a hypothesis:

1. **What** information does the technique represent?
2. **Why** could it improve ROC-AUC?
3. Is that information already represented by the baseline?
4. Can the idea be isolated in a controlled experiment?
5. Does the measured delta justify keeping it?

The baseline remains frozen. Each hypothesis is tested independently against the same feature matrix and the same validation folds.

## 15. V17 — Last-N installment events

### Hypothesis
Calendar windows such as 6M/12M/24M contain different numbers of events for different customers. Ranking events by recency and summarizing the **last 3, 5, and 10 payments** may preserve event-order information that fixed time windows lose.

### Result
**3-fold AUC: 0.792210**  
**Δ vs screening baseline: +0.000208**

**Decision:** 🟡 Keep as a candidate. The gain is small, but it is the first winner-inspired representation to add positive standalone signal.

In [ ]:
# ============================================================
# V17 — WINNER HYPOTHESIS #1
# LAST-N INSTALLMENT EVENTS
# ============================================================

def build_last_n_installment_features(
    installments_df,
    n_values=(3, 5, 10)
):

    inst = installments_df.copy()


    # ========================================================
    # 1. ROW-LEVEL BEHAVIOR
    # ========================================================

    inst["PAYMENT_DELAY"] = (
        inst["DAYS_ENTRY_PAYMENT"] -
        inst["DAYS_INSTALMENT"]
    )

    inst["LATE"] = (
        inst["PAYMENT_DELAY"] > 0
    ).astype("int8")

    inst["LATE_30"] = (
        inst["PAYMENT_DELAY"] > 30
    ).astype("int8")


    inst["UNDERPAID"] = (
        inst["AMT_PAYMENT"] <
        inst["AMT_INSTALMENT"]
    ).astype("int8")


    inst["PAYMENT_RATIO"] = np.where(

        inst["AMT_INSTALMENT"] > 0,

        inst["AMT_PAYMENT"] /
        inst["AMT_INSTALMENT"],

        np.nan
    )


    # ========================================================
    # 2. KEEP HISTORICAL EVENTS ONLY
    # ========================================================

    inst = inst[
        inst["DAYS_INSTALMENT"] < 0
    ].copy()


    # ========================================================
    # 3. SORT EVENTS BY RECENCY
    # ========================================================
    #
    # DAYS_INSTALMENT:
    #
    # -10  = recent
    # -100 = older
    # -500 = much older
    #
    # descending therefore gives:
    #
    # most recent
    # ↓
    # oldest
    # ========================================================

    inst = inst.sort_values(
        [
            "SK_ID_CURR",
            "DAYS_INSTALMENT"
        ],
        ascending=[
            True,
            False
        ]
    )


    # ========================================================
    # 4. EVENT RANK
    # ========================================================

    inst["EVENT_RANK"] = (
        inst
        .groupby("SK_ID_CURR")
        .cumcount()
        + 1
    )


    # ========================================================
    # 5. CREATE LAST-N AGGREGATIONS
    # ========================================================

    result = None


    for n in n_values:

        recent_n = inst[
            inst["EVENT_RANK"] <= n
        ]


        agg = (
            recent_n
            .groupby("SK_ID_CURR")
            .agg(

                **{

                    f"INST_LAST{n}_COUNT":
                        ("EVENT_RANK", "count"),

                    f"INST_LAST{n}_LATE_RATIO":
                        ("LATE", "mean"),

                    f"INST_LAST{n}_LATE30_RATIO":
                        ("LATE_30", "mean"),

                    f"INST_LAST{n}_MEAN_DELAY":
                        ("PAYMENT_DELAY", "mean"),

                    f"INST_LAST{n}_MAX_DELAY":
                        ("PAYMENT_DELAY", "max"),

                    f"INST_LAST{n}_UNDERPAY_RATIO":
                        ("UNDERPAID", "mean"),

                    f"INST_LAST{n}_MEAN_PAYMENT_RATIO":
                        ("PAYMENT_RATIO", "mean")
                }
            )
            .reset_index()
        )


        if result is None:

            result = agg

        else:

            result = result.merge(
                agg,
                on="SK_ID_CURR",
                how="outer"
            )


    result.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    return result

In [ ]:
# ============================================================
# CREATE V17 FEATURES
# ============================================================

v17_features = build_last_n_installment_features(
    installments,
    n_values=(3, 5, 10)
)


print("=" * 70)
print("V17 — LAST-N INSTALLMENT FEATURES")
print("=" * 70)

print(
    "Customers:",
    v17_features["SK_ID_CURR"].nunique()
)

print(
    "New features:",
    v17_features.shape[1] - 1
)

print(
    "Shape:",
    v17_features.shape
)


display(
    v17_features.head()
)

In [ ]:
# ============================================================
# EVENT COVERAGE
# ============================================================

print("=" * 70)
print("CUSTOMER EVENT COVERAGE")
print("=" * 70)

for n in [3, 5, 10]:

    col = f"INST_LAST{n}_COUNT"

    enough = (
        v17_features[col] >= n
    ).mean()

    print(
        f"Customers with >= {n:2d} events: "
        f"{enough * 100:.2f}%"
    )

In [ ]:
# ============================================================
# CREATE V17 EXPERIMENT MATRIX
# ============================================================

X_v17 = X_train_base.merge(

    v17_features,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"
)


print("=" * 70)
print("V17 MATRIX")
print("=" * 70)

print(
    "Baseline shape:",
    X_train_base.shape
)

print(
    "V17 shape:",
    X_v17.shape
)

print(
    "Features added:",
    X_v17.shape[1] -
    X_train_base.shape[1]
)

## 16. V18 — Multi-level installment aggregation

### Hypothesis
Direct `installment → customer` aggregation may hide how risky behavior is distributed across individual previous loans. V18 first summarizes each `SK_ID_PREV`, then summarizes the customer's collection of loans.

### Result
**3-fold AUC: 0.791907**  
**Δ: −0.000095**

**Decision:** ⚪ Park / reject for now. The hierarchy is conceptually useful, but it appears redundant with the already rich installment feature set.

In [ ]:
# ============================================================
# V18 — MULTI-LEVEL INSTALLMENT AGGREGATION
#
# installment rows
#       ↓
# SK_ID_PREV (previous loan)
#       ↓
# SK_ID_CURR (customer)
# ============================================================

def build_multilevel_installment_features(installments_df):

    inst = installments_df.copy()


    # ========================================================
    # 1. ROW-LEVEL PAYMENT BEHAVIOR
    # ========================================================

    inst["PAYMENT_DELAY"] = (
        inst["DAYS_ENTRY_PAYMENT"] -
        inst["DAYS_INSTALMENT"]
    )

    inst["LATE"] = (
        inst["PAYMENT_DELAY"] > 0
    ).astype("int8")

    inst["LATE_30"] = (
        inst["PAYMENT_DELAY"] > 30
    ).astype("int8")

    inst["UNDERPAID"] = (
        inst["AMT_PAYMENT"] <
        inst["AMT_INSTALMENT"]
    ).astype("int8")


    inst["PAYMENT_RATIO"] = np.where(

        inst["AMT_INSTALMENT"] > 0,

        inst["AMT_PAYMENT"] /
        inst["AMT_INSTALMENT"],

        np.nan
    )


    # Historical events only
    inst = inst[
        inst["DAYS_INSTALMENT"] < 0
    ].copy()


    # ========================================================
    # 2. LEVEL 1:
    # INSTALLMENTS → PREVIOUS LOAN
    # ========================================================

    loan_level = (
        inst
        .groupby(
            ["SK_ID_CURR", "SK_ID_PREV"]
        )
        .agg(

            LOAN_PAYMENT_COUNT=(
                "DAYS_INSTALMENT",
                "count"
            ),

            LOAN_LATE_RATIO=(
                "LATE",
                "mean"
            ),

            LOAN_LATE30_RATIO=(
                "LATE_30",
                "mean"
            ),

            LOAN_MEAN_DELAY=(
                "PAYMENT_DELAY",
                "mean"
            ),

            LOAN_MAX_DELAY=(
                "PAYMENT_DELAY",
                "max"
            ),

            LOAN_UNDERPAY_RATIO=(
                "UNDERPAID",
                "mean"
            ),

            LOAN_MEAN_PAYMENT_RATIO=(
                "PAYMENT_RATIO",
                "mean"
            )
        )
        .reset_index()
    )


    # ========================================================
    # 3. LEVEL 2:
    # PREVIOUS LOANS → CUSTOMER
    # ========================================================

    behavior_cols = [

        "LOAN_PAYMENT_COUNT",

        "LOAN_LATE_RATIO",
        "LOAN_LATE30_RATIO",

        "LOAN_MEAN_DELAY",
        "LOAN_MAX_DELAY",

        "LOAN_UNDERPAY_RATIO",

        "LOAN_MEAN_PAYMENT_RATIO"
    ]


    customer_level = (
        loan_level
        .groupby("SK_ID_CURR")[behavior_cols]
        .agg([
            "mean",
            "max",
            "min",
            "std"
        ])
    )


    # ========================================================
    # 4. FLATTEN COLUMN NAMES
    # ========================================================

    customer_level.columns = [

        f"INST_ML_{feature.replace('LOAN_', '')}_{stat.upper()}"

        for feature, stat
        in customer_level.columns
    ]


    customer_level = (
        customer_level
        .reset_index()
    )


    # ========================================================
    # 5. NUMBER OF PREVIOUS LOANS
    # ========================================================

    loan_count = (
        loan_level
        .groupby("SK_ID_CURR")
        ["SK_ID_PREV"]
        .nunique()
        .rename("INST_ML_LOAN_COUNT")
        .reset_index()
    )


    customer_level = (
        customer_level
        .merge(
            loan_count,
            on="SK_ID_CURR",
            how="left"
        )
    )


    # ========================================================
    # CLEAN
    # ========================================================

    customer_level.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    return customer_level, loan_level

In [ ]:
# ============================================================
# CREATE V18 FEATURES
# ============================================================

v18_features, v18_loan_level = (
    build_multilevel_installment_features(
        installments
    )
)


print("=" * 70)
print("V18 — MULTI-LEVEL INSTALLMENT FEATURES")
print("=" * 70)

print(
    "Loan-level rows:",
    len(v18_loan_level)
)

print(
    "Customers:",
    v18_features["SK_ID_CURR"].nunique()
)

print(
    "New features:",
    v18_features.shape[1] - 1
)

print(
    "Shape:",
    v18_features.shape
)

display(
    v18_features.head()
)

In [ ]:
# ============================================================
# V18 EXPERIMENT MATRIX
# Baseline + V18 ONLY
# ============================================================

X_v18 = X_train_base.merge(

    v18_features,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"
)


print("=" * 70)
print("V18 MATRIX")
print("=" * 70)

print(
    "Baseline:",
    X_train_base.shape
)

print(
    "V18:",
    X_v18.shape
)

print(
    "Features added:",
    X_v18.shape[1] -
    X_train_base.shape[1]
)

## 17. V19 — Conditional previous-application aggregation

### Hypothesis
Global approval/refusal rates may hide the characteristics of applications *within* each state. V19 separately aggregates approved and refused previous applications and compares them.

### Result
**3-fold AUC: 0.791827**  
**Δ: −0.000175**

**Decision:** 🔴 Reject. Conditional summaries did not add incremental ranking power on top of the existing previous-application features.

In [ ]:
# ============================================================
# V19 — CONDITIONAL PREVIOUS-APPLICATION AGGREGATION
# Approved vs Refused
# ============================================================

def build_conditional_previous_features(prev_df):

    prev = prev_df.copy()


    # ========================================================
    # 1. ROW-LEVEL FEATURES
    # ========================================================

    prev["PREV_DECISION_AGE"] = (
        -prev["DAYS_DECISION"]
    )


    prev["PREV_CREDIT_APPLICATION_RATIO"] = (
        prev["AMT_CREDIT"] /
        prev["AMT_APPLICATION"].replace(0, np.nan)
    )


    prev["PREV_CREDIT_APPLICATION_RATIO"] = (
        prev["PREV_CREDIT_APPLICATION_RATIO"]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )


    # ========================================================
    # 2. CONDITIONAL AGGREGATION FUNCTION
    # ========================================================

    def aggregate_status(
        df,
        status,
        prefix
    ):

        temp = df[
            df["NAME_CONTRACT_STATUS"] == status
        ].copy()


        agg = (
            temp
            .groupby("SK_ID_CURR")
            .agg(

                **{
                    # Number of applications in this state
                    f"{prefix}_COUNT":
                        ("SK_ID_PREV", "count"),


                    # Requested amount
                    f"{prefix}_APPLICATION_MEAN":
                        ("AMT_APPLICATION", "mean"),

                    f"{prefix}_APPLICATION_MAX":
                        ("AMT_APPLICATION", "max"),


                    # Granted / proposed credit
                    f"{prefix}_CREDIT_MEAN":
                        ("AMT_CREDIT", "mean"),

                    f"{prefix}_CREDIT_MAX":
                        ("AMT_CREDIT", "max"),


                    # Annuity
                    f"{prefix}_ANNUITY_MEAN":
                        ("AMT_ANNUITY", "mean"),


                    # Granted / requested relationship
                    f"{prefix}_CREDIT_APP_RATIO_MEAN":
                        (
                            "PREV_CREDIT_APPLICATION_RATIO",
                            "mean"
                        ),


                    # Historical timing
                    f"{prefix}_DECISION_AGE_MEAN":
                        ("PREV_DECISION_AGE", "mean"),

                    f"{prefix}_DECISION_AGE_MIN":
                        ("PREV_DECISION_AGE", "min")
                }
            )
            .reset_index()
        )

        return agg


    # ========================================================
    # 3. APPROVED APPLICATIONS
    # ========================================================

    approved = aggregate_status(

        prev,

        status="Approved",

        prefix="PREV_COND_APPROVED"
    )


    # ========================================================
    # 4. REFUSED APPLICATIONS
    # ========================================================

    refused = aggregate_status(

        prev,

        status="Refused",

        prefix="PREV_COND_REFUSED"
    )


    # ========================================================
    # 5. COMBINE BOTH STATES
    # ========================================================

    conditional = approved.merge(

        refused,

        on="SK_ID_CURR",

        how="outer"
    )


    # ========================================================
    # 6. CROSS-STATUS DIFFERENCES
    # ========================================================

    # Did refused applications request more money?
    conditional[
        "PREV_COND_APPLICATION_MEAN_DIFF"
    ] = (

        conditional[
            "PREV_COND_REFUSED_APPLICATION_MEAN"
        ]

        -

        conditional[
            "PREV_COND_APPROVED_APPLICATION_MEAN"
        ]
    )


    # Difference in granted/requested relationship
    conditional[
        "PREV_COND_CREDIT_APP_RATIO_DIFF"
    ] = (

        conditional[
            "PREV_COND_REFUSED_CREDIT_APP_RATIO_MEAN"
        ]

        -

        conditional[
            "PREV_COND_APPROVED_CREDIT_APP_RATIO_MEAN"
        ]
    )


    # Recency difference
    #
    # Smaller age = more recent.
    conditional[
        "PREV_COND_DECISION_AGE_DIFF"
    ] = (

        conditional[
            "PREV_COND_REFUSED_DECISION_AGE_MIN"
        ]

        -

        conditional[
            "PREV_COND_APPROVED_DECISION_AGE_MIN"
        ]
    )


    conditional.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )


    return conditional

In [ ]:
# ============================================================
# CREATE V19 FEATURES
# ============================================================

v19_features = (
    build_conditional_previous_features(
        previous_application
    )
)


print("=" * 70)
print("V19 — CONDITIONAL PREVIOUS APPLICATION FEATURES")
print("=" * 70)

print(
    "Customers:",
    v19_features["SK_ID_CURR"].nunique()
)

print(
    "New features:",
    v19_features.shape[1] - 1
)

print(
    "Shape:",
    v19_features.shape
)


display(
    v19_features.head()
)

In [ ]:
# ============================================================
# V19 EXPERIMENT MATRIX
#
# Baseline + V19 ONLY
# ============================================================

X_v19 = X_train_base.merge(

    v19_features,

    on="SK_ID_CURR",

    how="left",

    validate="one_to_one"
)


print("=" * 70)
print("V19 MATRIX")
print("=" * 70)

print(
    "Baseline:",
    X_train_base.shape
)

print(
    "V19:",
    X_v19.shape
)

print(
    "Features added:",
    X_v19.shape[1] -
    X_train_base.shape[1]
)

## 18. Winner-study experiment leaderboard

| Experiment | Technique | Added features | 3-fold AUC | Δ vs baseline | Decision |
|---|---|---:|---:|---:|---|
| Baseline | Reconstructed V13 | — | 0.792002 | — | 🔒 Reference |
| V17 | Last-N installment events | +21 | **0.792210** | **+0.000208** | 🟡 Keep |
| V18 | Multi-level installments | +29 | 0.791907 | −0.000095 | ⚪ Park |
| V19 | Conditional previous applications | +21 | 0.791827 | −0.000175 | 🔴 Reject |

### Interpretation
The first three winner-inspired experiments suggest an important pattern:

- reorganizing information that the baseline already summarizes well (V18, V19) produced little or negative incremental value;
- introducing a **new representation of recency through event order** (V17) produced a small positive signal.

This distinction is central to the project: a technique is valuable only when it contributes information not already captured by the existing pipeline.

## 19. Key lessons learned

### Feature engineering
- Relational tables were the main source of improvement.
- Recent behavior was more informative than simply adding more lifetime aggregates.
- Extreme ratio features should be treated carefully; clipping and signed-log transforms can stabilize them.
- Cross-table interactions were mostly marginal once strong component features already existed.

### Experimental methodology
- Keep validation splits fixed when comparing feature families.
- Test one hypothesis at a time.
- Record negative results; they prevent repeated work and reveal redundancy.
- A winning technique is not automatically useful on top of a different pipeline.

### Modeling
LightGBM is a strong fit for this heterogeneous tabular dataset because it handles nonlinear interactions, missing values, mixed feature scales, and high-dimensional engineered feature spaces effectively.

### Next research direction
The most promising current clue is **event-order recency**. A logical next experiment is to test Last-N / lag representations on other relational tables before moving to ensembling.

## 20. Reproducibility and GitHub notes

Recommended repository files:

```text
home-credit-default-risk/
├── README.md
├── home_credit_default_risk.ipynb
├── requirements.txt
├── .gitignore
└── data/                  # local only; ignored by Git
```

Suggested `.gitignore` entries:

```text
data/
*.csv
.ipynb_checkpoints/
__pycache__/
```

The competition data should be obtained from Kaggle rather than committed to the repository.